# Aula 07 - Compressão e Descompressão de Arquivos
## ZIP, GZIP, RAR, TAR.GZ e TAR.BZ2

**QXD0099 - Desenvolvimento de Software para Persistência**  
Universidade Federal do Ceará - Campus Quixadá  
Prof. Francisco Victor da Silva Pinheiro  
victorpinheiro@ufc.br

> Notebook prático e comentado para execução em sala no Google Colab.

## Agenda
- Por que comprimir arquivos
- Compressão sem perda e com perda
- ZIP
- GZIP
- TAR
- TAR.GZ
- RAR
- TAR.BZ2
- `zipfile`
- `gzip`
- `tarfile`
- `bz2`
- Comparação de tamanhos
- Mini gerenciador de backup

# 1. Por que comprimir arquivos?

- economia de espaço;
- transferência mais rápida;
- agrupamento de arquivos;
- backups e distribuição de pacotes.

**Arquivamento** reúne arquivos.  
**Compressão** reduz o número de bytes utilizados.

# 2. Tipos de compressão

## Sem perda
Preserva os dados originais: ZIP, GZIP, BZIP2, TAR.GZ.

## Com perda
Descarta informação irreversivelmente: JPEG, MP3, MP4.

# 3. Principais formatos

| Formato | Característica |
|---|---|
| ZIP | Agrupa e comprime múltiplos arquivos |
| GZIP | Normalmente comprime um arquivo/fluxo |
| TAR | Agrupa arquivos |
| TAR.GZ | TAR + GZIP |
| TAR.BZ2 | TAR + BZIP2 |
| RAR | Formato proprietário |
| 7z | Alta compressão |

# 4. Preparando arquivos de exemplo

In [ ]:
from pathlib import Path

pasta = Path("arquivos_exemplo")
pasta.mkdir(exist_ok=True)

(pasta / "arquivo1.txt").write_text(
    "Persistência de Arquivos - QXD0099\n" * 100,
    encoding="utf-8"
)

(pasta / "arquivo2.txt").write_text(
    "Universidade Federal do Ceará - Campus Quixadá\n" * 80,
    encoding="utf-8"
)

(pasta / "arquivo3.txt").write_text(
    "Compressão, arquivamento e persistência em Python.\n" * 120,
    encoding="utf-8"
)

for arquivo in pasta.iterdir():
    print(arquivo.name, "->", arquivo.stat().st_size, "bytes")

arquivo3.txt -> 6360 bytes
arquivo2.txt -> 3920 bytes
arquivo1.txt -> 3600 bytes


# 5. ZIP pelo terminal

In [ ]:
!zip -r arquivos_terminal.zip arquivos_exemplo

  adding: arquivos_exemplo/ (stored 0%)
  adding: arquivos_exemplo/arquivo3.txt (deflated 99%)
  adding: arquivos_exemplo/arquivo2.txt (deflated 98%)
  adding: arquivos_exemplo/arquivo1.txt (deflated 98%)


In [ ]:
!unzip -l arquivos_terminal.zip

Archive:  arquivos_terminal.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-09-04 12:18   arquivos_exemplo/
     6360  2026-09-04 12:18   arquivos_exemplo/arquivo3.txt
     3920  2026-09-04 12:18   arquivos_exemplo/arquivo2.txt
     3600  2026-09-04 12:18   arquivos_exemplo/arquivo1.txt
---------                     -------
    13880                     4 files


In [ ]:
!mkdir -p zip_terminal_extraido
!unzip -o arquivos_terminal.zip -d zip_terminal_extraido

Archive:  arquivos_terminal.zip
   creating: zip_terminal_extraido/arquivos_exemplo/
  inflating: zip_terminal_extraido/arquivos_exemplo/arquivo3.txt  
  inflating: zip_terminal_extraido/arquivos_exemplo/arquivo2.txt  
  inflating: zip_terminal_extraido/arquivos_exemplo/arquivo1.txt  


# 6. ZIP com Python

In [ ]:
import zipfile

with zipfile.ZipFile(
    "arquivos_python.zip",
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zipf:
    for arquivo in pasta.iterdir():
        zipf.write(arquivo, arcname=arquivo.name)

print("ZIP criado.")

ZIP criado.


## 7. Listando conteúdo do ZIP

In [ ]:
with zipfile.ZipFile("arquivos_python.zip", "r") as zipf:
    for nome in zipf.namelist():
        print(nome)

arquivo3.txt
arquivo2.txt
arquivo1.txt


## 8. Informações de compressão

In [ ]:
with zipfile.ZipFile("arquivos_python.zip", "r") as zipf:
    for info in zipf.infolist():
        print("Nome:", info.filename)
        print("Original:", info.file_size, "bytes")
        print("Comprimido:", info.compress_size, "bytes")

        if info.file_size:
            reducao = (1 - info.compress_size / info.file_size) * 100
            print("Redução:", round(reducao, 2), "%")

        print("---")

Nome: arquivo3.txt
Original: 6360 bytes
Comprimido: 93 bytes
Redução: 98.54 %
---
Nome: arquivo2.txt
Original: 3920 bytes
Comprimido: 78 bytes
Redução: 98.01 %
---
Nome: arquivo1.txt
Original: 3600 bytes
Comprimido: 69 bytes
Redução: 98.08 %
---


## 9. Extraindo ZIP

In [ ]:
from pathlib import Path
import zipfile

destino = Path("zip_python_extraido")
destino.mkdir(exist_ok=True)

with zipfile.ZipFile("arquivos_python.zip", "r") as zipf:
    zipf.extractall(destino)

print("Extraído em:", destino)

Extraído em: zip_python_extraido


## 10. Lendo arquivo dentro do ZIP sem extrair

In [ ]:
with zipfile.ZipFile("arquivos_python.zip", "r") as zipf:
    with zipf.open("arquivo1.txt") as arquivo:
        texto = arquivo.read().decode("utf-8")
        print(texto[:300])

Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arquivos - QXD0099
Persistência de Arqu


# 11. GZIP

Terminal:

```bash
gzip arquivo.txt
gunzip arquivo.txt.gz
```

In [ ]:
!cp arquivos_exemplo/arquivo1.txt arquivo_gzip.txt
!gzip arquivo_gzip.txt
!ls -lh arquivo_gzip.txt.gz

-rw-r--r-- 1 root root 104 Sep  4 00:01 arquivo_gzip.txt.gz


In [ ]:
!gunzip arquivo_gzip.txt.gz
!ls -lh arquivo_gzip.txt

-rw-r--r-- 1 root root 3.6K Sep  4 00:01 arquivo_gzip.txt


# 12. GZIP com Python

In [ ]:
import gzip
import shutil

with open("arquivos_exemplo/arquivo1.txt", "rb") as origem:
    with gzip.open("arquivo1_python.txt.gz", "wb") as destino:
        shutil.copyfileobj(origem, destino)

print("GZIP criado.")

GZIP criado.


## 13. Descompactando GZIP

In [ ]:
with gzip.open("arquivo1_python.txt.gz", "rb") as origem:
    with open("arquivo1_restaurado.txt", "wb") as destino:
        shutil.copyfileobj(origem, destino)

print("Arquivo restaurado.")

Arquivo restaurado.


## 14. Verificando integridade com SHA-256

In [ ]:
import hashlib

def sha256(arquivo):
    h = hashlib.sha256()

    with open(arquivo, "rb") as f:
        for bloco in iter(lambda: f.read(65536), b""):
            h.update(bloco)

    return h.hexdigest()

original = sha256("arquivos_exemplo/arquivo1.txt")
restaurado = sha256("arquivo1_restaurado.txt")

print("Original:  ", original)
print("Restaurado:", restaurado)
print("Idênticos?", original == restaurado)

Original:   87f589d1d467a1bfb7fb38a85247965aa84d1907a55b924cab0f5266722744af
Restaurado: 87f589d1d467a1bfb7fb38a85247965aa84d1907a55b924cab0f5266722744af
Idênticos? True


# 15. TAR

TAR agrupa arquivos e não precisa, por si só, aplicar compressão.

In [ ]:
!tar -cvf arquivos.tar arquivos_exemplo

arquivos_exemplo/
arquivos_exemplo/arquivo3.txt
arquivos_exemplo/arquivo2.txt
arquivos_exemplo/arquivo1.txt


In [ ]:
!tar -tvf arquivos.tar

drwxr-xr-x root/root         0 2026-09-04 00:01 arquivos_exemplo/
-rw-r--r-- root/root      6360 2026-09-04 00:01 arquivos_exemplo/arquivo3.txt
-rw-r--r-- root/root      3920 2026-09-04 00:01 arquivos_exemplo/arquivo2.txt
-rw-r--r-- root/root      3600 2026-09-04 00:01 arquivos_exemplo/arquivo1.txt


# 16. TAR.GZ pelo terminal

In [ ]:
!tar -czvf arquivos.tar.gz arquivos_exemplo

arquivos_exemplo/
arquivos_exemplo/arquivo3.txt
arquivos_exemplo/arquivo2.txt
arquivos_exemplo/arquivo1.txt


In [ ]:
!mkdir -p targz_extraido
!tar -xzvf arquivos.tar.gz -C targz_extraido

arquivos_exemplo/
arquivos_exemplo/arquivo3.txt
arquivos_exemplo/arquivo2.txt
arquivos_exemplo/arquivo1.txt


# 17. TAR.GZ com Python

In [ ]:
import tarfile

with tarfile.open("arquivos_python.tar.gz", "w:gz") as tar:
    tar.add("arquivos_exemplo", arcname="arquivos_exemplo")

print("TAR.GZ criado.")

TAR.GZ criado.


In [ ]:
with tarfile.open("arquivos_python.tar.gz", "r:gz") as tar:
    for membro in tar.getmembers():
        print(membro.name)

arquivos_exemplo
arquivos_exemplo/arquivo1.txt
arquivos_exemplo/arquivo2.txt
arquivos_exemplo/arquivo3.txt


In [ ]:
from pathlib import Path
import tarfile

destino = Path("targz_python_extraido")
destino.mkdir(exist_ok=True)

with tarfile.open("arquivos_python.tar.gz", "r:gz") as tar:
    tar.extractall(destino)

print("Extraído em:", destino)

Extraído em: targz_python_extraido


/tmp/ipykernel_1271/1956711017.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(destino)


# 18. TAR.BZ2 pelo terminal

In [ ]:
!tar -cjvf arquivos.tar.bz2 arquivos_exemplo

arquivos_exemplo/
arquivos_exemplo/arquivo3.txt
arquivos_exemplo/arquivo2.txt
arquivos_exemplo/arquivo1.txt


In [ ]:
!mkdir -p tarbz2_extraido
!tar -xjvf arquivos.tar.bz2 -C tarbz2_extraido

arquivos_exemplo/
arquivos_exemplo/arquivo3.txt
arquivos_exemplo/arquivo2.txt
arquivos_exemplo/arquivo1.txt


# 19. TAR.BZ2 com Python

In [ ]:
import tarfile

with tarfile.open("arquivos_python.tar.bz2", "w:bz2") as tar:
    tar.add("arquivos_exemplo", arcname="arquivos_exemplo")

print("TAR.BZ2 criado.")

TAR.BZ2 criado.


In [ ]:
destino = Path("tarbz2_python_extraido")
destino.mkdir(exist_ok=True)

with tarfile.open("arquivos_python.tar.bz2", "r:bz2") as tar:
    tar.extractall(destino)

print("Extraído em:", destino)

Extraído em: tarbz2_python_extraido


/tmp/ipykernel_1271/870037569.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(destino)


# 20. Compressão direta com bz2

In [ ]:
import bz2

conteudo = ("Conteúdo do arquivo usando BZIP2.\n" * 100).encode("utf-8")

with bz2.BZ2File("arquivo_texto.txt.bz2", "w") as arquivo:
    arquivo.write(conteudo)

print("BZ2 criado.")

BZ2 criado.


## 21. Descompactando com bz2

In [ ]:
with bz2.BZ2File("arquivo_texto.txt.bz2", "r") as arquivo:
    dados = arquivo.read()

print(dados.decode("utf-8")[:300])

Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando BZIP2.
Conteúdo do arquivo usando B


## 22. Compressão em memória

In [ ]:
import bz2

dados = ("Persistência de Arquivos " * 100).encode("utf-8")

comprimido = bz2.compress(dados)
restaurado = bz2.decompress(comprimido)

print("Original:", len(dados), "bytes")
print("Comprimido:", len(comprimido), "bytes")
print("Restaurado corretamente?", restaurado == dados)

Original: 2600 bytes
Comprimido: 103 bytes
Restaurado corretamente? True


# 23. RAR

RAR normalmente depende de software externo.

Comandos apresentados na aula:

```bash
rar a arquivo.rar arquivo1.txt arquivo2.txt
unrar x arquivo.rar
```

A biblioteca `rarfile` pode ler/extrair arquivos RAR, mas depende de ferramentas externas em muitos ambientes.

In [ ]:
!pip -q install rarfile

In [ ]:
import shutil

print("rar:", shutil.which("rar"))
print("unrar:", shutil.which("unrar"))

rar: None
unrar: /usr/bin/unrar


Se `rar` estiver disponível:

```python
import os
os.system("rar a arquivo.rar arquivo1.txt arquivo2.txt")
```

Para extração:

```python
import rarfile

with rarfile.RarFile("arquivo.rar") as rar:
    rar.extractall("rar_extraido")
```

# 24. Comparando tamanhos

In [ ]:
from pathlib import Path

arquivos = [
    "arquivos.tar",
    "arquivos_terminal.zip",
    "arquivos_python.zip",
    "arquivos.tar.gz",
    "arquivos_python.tar.gz",
    "arquivos.tar.bz2",
    "arquivos_python.tar.bz2"
]

for nome in arquivos:
    p = Path(nome)

    if p.exists():
        print(f"{nome:28} {p.stat().st_size:8} bytes")

arquivos.tar                    20480 bytes
arquivos_terminal.zip             982 bytes
arquivos_python.zip               562 bytes
arquivos.tar.gz                   377 bytes
arquivos_python.tar.gz            500 bytes
arquivos.tar.bz2                  466 bytes
arquivos_python.tar.bz2           584 bytes


## 25. Original x ZIP

In [ ]:
tamanho_original = sum(
    arquivo.stat().st_size
    for arquivo in pasta.iterdir()
)

tamanho_zip = Path("arquivos_python.zip").stat().st_size

reducao = (1 - tamanho_zip / tamanho_original) * 100

print("Original:", tamanho_original, "bytes")
print("ZIP:", tamanho_zip, "bytes")
print("Redução:", round(reducao, 2), "%")

Original: 13880 bytes
ZIP: 562 bytes
Redução: 95.95 %


# 26. Função para criar backup ZIP

In [ ]:
import zipfile
from pathlib import Path

def criar_backup_zip(pasta_origem, arquivo_destino):

    pasta_origem = Path(pasta_origem)

    with zipfile.ZipFile(
        arquivo_destino,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as zipf:

        for arquivo in pasta_origem.rglob("*"):

            if arquivo.is_file():

                zipf.write(
                    arquivo,
                    arcname=arquivo.relative_to(pasta_origem)
                )

    print("Backup criado:", arquivo_destino)

In [ ]:
criar_backup_zip("arquivos_exemplo", "backup.zip")

Backup criado: backup.zip


# 27. Restaurando backup ZIP

In [ ]:
def restaurar_backup_zip(arquivo_zip, pasta_destino):

    destino = Path(pasta_destino)
    destino.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(arquivo_zip, "r") as zipf:
        zipf.extractall(destino)

    print("Backup restaurado em:", destino)

In [ ]:
restaurar_backup_zip("backup.zip", "backup_restaurado")

Backup restaurado em: backup_restaurado


# 28. Atividade prática — Gerenciador de Backup

Implemente:

```text
1 - Criar backup ZIP
2 - Listar conteúdo
3 - Extrair backup
4 - Mostrar tamanho original
5 - Mostrar tamanho compactado
0 - Sair
```

Use:

```text
documentos/
backup_documentos.zip
```

# 29. Solução de referência

In [ ]:
import zipfile
from pathlib import Path

PASTA_DOCUMENTOS = Path("documentos")
ARQUIVO_BACKUP = Path("backup_documentos.zip")

PASTA_DOCUMENTOS.mkdir(exist_ok=True)

for i in range(1, 4):
    arquivo = PASTA_DOCUMENTOS / f"doc{i}.txt"

    if not arquivo.exists():
        arquivo.write_text(
            (f"Documento {i} - QXD0099\n" * 100),
            encoding="utf-8"
        )


def criar_backup():
    with zipfile.ZipFile(
        ARQUIVO_BACKUP,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as zipf:

        for arquivo in PASTA_DOCUMENTOS.rglob("*"):

            if arquivo.is_file():
                zipf.write(
                    arquivo,
                    arcname=arquivo.relative_to(PASTA_DOCUMENTOS)
                )

    print("Backup criado.")


def listar_backup():
    if not ARQUIVO_BACKUP.exists():
        print("Backup ainda não existe.")
        return

    with zipfile.ZipFile(ARQUIVO_BACKUP, "r") as zipf:
        for nome in zipf.namelist():
            print("-", nome)


def extrair_backup():
    if not ARQUIVO_BACKUP.exists():
        print("Backup ainda não existe.")
        return

    destino = Path("documentos_restaurados")
    destino.mkdir(exist_ok=True)

    with zipfile.ZipFile(ARQUIVO_BACKUP, "r") as zipf:
        zipf.extractall(destino)

    print("Backup restaurado em:", destino)


def tamanho_original():
    total = sum(
        arquivo.stat().st_size
        for arquivo in PASTA_DOCUMENTOS.rglob("*")
        if arquivo.is_file()
    )

    print("Tamanho original:", total, "bytes")


def tamanho_backup():
    if not ARQUIVO_BACKUP.exists():
        print("Backup ainda não existe.")
        return

    print(
        "Tamanho do backup:",
        ARQUIVO_BACKUP.stat().st_size,
        "bytes"
    )


while True:

    print("\n=== GERENCIADOR DE BACKUP ===")
    print("1 - Criar backup ZIP")
    print("2 - Listar conteúdo")
    print("3 - Extrair backup")
    print("4 - Tamanho original")
    print("5 - Tamanho compactado")
    print("0 - Sair")

    opcao = input("Opção: ")

    if opcao == "1":
        criar_backup()
    elif opcao == "2":
        listar_backup()
    elif opcao == "3":
        extrair_backup()
    elif opcao == "4":
        tamanho_original()
    elif opcao == "5":
        tamanho_backup()
    elif opcao == "0":
        print("Programa encerrado.")
        break
    else:
        print("Opção inválida.")


=== GERENCIADOR DE BACKUP ===
1 - Criar backup ZIP
2 - Listar conteúdo
3 - Extrair backup
4 - Tamanho original
5 - Tamanho compactado
0 - Sair
Opção: 0
Programa encerrado.


# Fechamento

```text
Arquivos
   ↓
Arquivamento / Compressão
   ├── ZIP
   ├── GZIP
   ├── TAR
   ├── TAR.GZ
   ├── TAR.BZ2
   ├── BZ2
   └── RAR
```

## Pontos principais

- compressão sem perda preserva os bytes originais;
- ZIP agrupa e comprime;
- GZIP comprime um fluxo/arquivo;
- TAR arquiva;
- TAR.GZ = TAR + GZIP;
- TAR.BZ2 = TAR + BZIP2;
- `zipfile`, `gzip`, `tarfile` e `bz2` são bibliotecas padrão úteis;
- RAR depende de ferramenta externa;
- hashes ajudam a confirmar integridade após descompressão;
- backups são uma aplicação direta desses conceitos.